# SAT-guided Turyn dataset (partial, next bit)

1. Build a **random partial** with a target **flat length** (total assigned ±1 symbols).
2. **Extend to a full length‑`r` assignment** by repeatedly proposing the next cell (same schedule as `_add_bit_to_seq`), testing **+1** and **−1** with the SAT encoding.
3. **Canonicalize** the completed sequence.
4. Emit the same **`(partial, next_bit)`** pairs as `sequence_generator.generate_dataset` via `create_partials` / `get_partial`.

In [1]:
import copy
import random

from cnf_automator import generate_encoding, verify_assignment
from sequence_generator import (
    canonicalize_seq,
    create_partials,
    create_partial_str_next_bit,
    is_turyn,
)

print("Imports OK")

Imports OK


## Parameters

- `r`: Turyn row length (all four rows length `r` when complete).
- `partial_flat_len`: total number of ±1 symbols in the random partial **after** seeding the first column with four `1`s (so minimum `4`).
- Must satisfy `4 <= partial_flat_len < 4 * r` so completion is nontrivial.

In [2]:
R = 7
PARTIAL_FLAT_LEN = 12
RNG_SEED = 42

random.seed(RNG_SEED)

assert 4 <= PARTIAL_FLAT_LEN < 4 * R, f"Need 4 <= partial_flat_len < {4 * R}"

cnf, varmap, pmap_lags, pool = generate_encoding(R)
print(f"r={R}, partial_flat_len={PARTIAL_FLAT_LEN}, CNF clauses={len(cnf.clauses)}")

r=7, partial_flat_len=12, CNF clauses=2400


## Bit order (matches `turyn_policy_transformer._add_bit_to_seq`)

Same rule as SAT rollout: seed each row with one `1`, then always append to the first row strictly shorter than row 0, else append to row 0.

In [5]:
def next_row_for_bit(seq):
    """Which row receives the next symbol (lists or tuples of ±1)."""
    first_len = len(seq[0])
    for i in range(4):
        if len(seq[i]) < first_len:
            return i
    return 0


def add_bit_to_seq(seq, bit):
    """seq: list of lists (mutable). bit is +1 or -1."""
    row = next_row_for_bit(seq)
    seq[row].append(bit)


def flat_len(seq):
    return sum(len(row) for row in seq)


def seq_to_tuples(seq):
    return tuple(tuple(row) for row in seq)


def random_partial_sequence(r, partial_flat_len, rng=random):
    """
    Random partial: four rows each start with +1, then random ±1 bits until
    sum(len(row)) == partial_flat_len.
    """
    if partial_flat_len < 4 or partial_flat_len >= 4 * r:
        raise ValueError(f"partial_flat_len must be in [4, 4*r), got {partial_flat_len}")
    seq = [[1], [1], [1], [1]]
    while flat_len(seq) < partial_flat_len:
        add_bit_to_seq(seq, 1 if rng.random() < 0.5 else -1)
    return seq_to_tuples(seq)


# Demo shape
demo_p = random_partial_sequence(R, PARTIAL_FLAT_LEN)
print("Random partial (row lengths):", [len(x) for x in demo_p])
print("Flat length:", flat_len(demo_p))
for i, row in enumerate(demo_p):
    print(f"  row {i}: {row}")

Random partial (row lengths): [3, 3, 3, 3]
Flat length: 12
  row 0: (1, 1, -1)
  row 1: (1, -1, -1)
  row 2: (1, -1, 1)
  row 3: (1, 1, 1)


## SAT-guided completion

At each step we **query SAT for both** `−1` and `+1` at the next cell (see printed `sat(-1)` / `sat(+1)`). We then commit to the first feasible value in a **shuffled** order so completions stay diverse. If neither extends, return `None` so the caller can resample.

In [6]:
def extend_to_full_turyn_sat(partial_tuples, r, varmap, cnf, rng=random, verbose=False):
    """
    Greedy SAT extension: same append order as random_partial / rollout.
    Returns full sequence as 4 tuples of length r, or None.
    """
    seq = [list(t) for t in partial_tuples]
    if any(len(row) > r for row in seq):
        return None
    step = 0
    while any(len(row) < r for row in seq):
        row = next_row_for_bit(seq)
        sat_by_bit = {}
        for bit in (-1, 1):
            trial = copy.deepcopy(seq)
            trial[row].append(bit)
            if any(len(x) > r for x in trial):
                sat_by_bit[bit] = False
            else:
                sat_by_bit[bit], _ = verify_assignment(
                    seq_to_tuples(trial), varmap, cnf
                )
        if verbose:
            print(
                f"step {step}: append to row {row} | "
                f"sat(-1)={sat_by_bit[-1]} sat(+1)={sat_by_bit[1]}"
            )
            step += 1
        order = [-1, 1]
        rng.shuffle(order)
        chosen = None
        for bit in order:
            if sat_by_bit.get(bit):
                chosen = bit
                break
        if chosen is None:
            return None
        seq[row].append(chosen)
    return seq_to_tuples(seq)


# One completion attempt (may retry a few times if random partial is dead)
attempt = 0
full_seq = None
last_partial = None
show_steps_once = True
while full_seq is None and attempt < 30:
    last_partial = random_partial_sequence(R, PARTIAL_FLAT_LEN)
    sat0, _ = verify_assignment(last_partial, varmap, cnf)
    print(
        f"try {attempt + 1}: row lens {[len(x) for x in last_partial]} "
        f"partial globally extendable? {sat0}"
    )
    do_verbose = show_steps_once and sat0
    if do_verbose:
        print("  (verbose) both bits queried each step:")
    full_seq = extend_to_full_turyn_sat(
        last_partial,
        R,
        varmap,
        cnf,
        rng=random,
        verbose=do_verbose,
    )
    if full_seq is not None:
        show_steps_once = False
    attempt += 1

if full_seq is None:
    raise RuntimeError("Could not complete after 30 random partials; try smaller PARTIAL_FLAT_LEN or different R")

print("Completed rows:")
for i, row in enumerate(full_seq):
    print(f"  row {i}: {row}")
print("is_turyn:", is_turyn(R, full_seq))

try 1: row lens [3, 3, 3, 3] partial globally extendable? True
  (verbose) both bits queried each step:
step 0: append to row 0 | sat(-1)=True sat(+1)=True
step 1: append to row 1 | sat(-1)=False sat(+1)=True
step 2: append to row 2 | sat(-1)=False sat(+1)=True
step 3: append to row 3 | sat(-1)=False sat(+1)=True
step 4: append to row 0 | sat(-1)=False sat(+1)=True
step 5: append to row 1 | sat(-1)=True sat(+1)=True
step 6: append to row 2 | sat(-1)=True sat(+1)=False
step 7: append to row 3 | sat(-1)=True sat(+1)=False
step 8: append to row 0 | sat(-1)=False sat(+1)=True
step 9: append to row 1 | sat(-1)=False sat(+1)=True
step 10: append to row 2 | sat(-1)=True sat(+1)=True
step 11: append to row 3 | sat(-1)=True sat(+1)=False
step 12: append to row 0 | sat(-1)=False sat(+1)=True
step 13: append to row 1 | sat(-1)=True sat(+1)=False
step 14: append to row 2 | sat(-1)=False sat(+1)=True
step 15: append to row 3 | sat(-1)=True sat(+1)=False
Completed rows:
  row 0: (1, -1, -1, 1, 1, 1,

## Canonicalize + same `(partial, next_bit)` pairs as `generate_dataset`

In [ ]:
# canonicalize_seq mutates a list of rows in place
canon = [list(row) for row in full_seq]
canonicalize_seq(canon)
canon_tuples = seq_to_tuples(canon)

print("Canonical full sequence:")
for i, row in enumerate(canon_tuples):
    print(f"  row {i}: {row}")

partials_from_canon = create_partials(canon_tuples, R)
print(f"\nNumber of (partial, next_bit) from create_partials: {len(partials_from_canon)}")

for k in range(min(3, len(partials_from_canon))):
    p, nb = partials_from_canon[k]
    s = create_partial_str_next_bit(p, nb)
    print(f"\nExample[{k}] next_bit={nb:+d} key[:40]...={s[:40]!r}")
    for i, row in enumerate(p):
        print(f"    row {i} len={len(row)}: {row}")

Canonical full sequence:
  row 0: (1, -1, -1, 1, 1, 1, 1)
  row 1: (1, 1, -1, 1, 1, 1, -1)
  row 2: (1, 1, -1, 1, -1, 1, 1)
  row 3: (-1, -1, -1, 1, -1, 1, 1)

Number of (partial, next_bit) from create_partials: 28

Example[0] next_bit=+1 key[:40]...='1'
    row 0 len=0: ()
    row 1 len=0: ()
    row 2 len=0: ()
    row 3 len=0: ()

Example[1] next_bit=+1 key[:40]...='11'
    row 0 len=1: (1,)
    row 1 len=0: ()
    row 2 len=0: ()
    row 3 len=0: ()

Example[2] next_bit=+1 key[:40]...='111'
    row 0 len=1: (1,)
    row 1 len=1: (1,)
    row 2 len=0: ()
    row 3 len=0: ()


## Dataset builder: many trajectories, dedupe like `generate_dataset`

In [ ]:
from sequence_generator_sat import create_partials_excluding_seed


def generate_dataset_sat(
    r,
    partial_flat_len,
    num_successful_sequences=5,
    max_partial_tries_per_seq=80,
    rng=random,
):
    """
    For each successful completion:
      SAT-complete random partial -> partials from raw full seq (no canonicalize).
    Merge and dedupe by create_partial_str_next_bit (same as generate_dataset).
    """
    cnf, varmap, _, _ = generate_encoding(r)
    dataset = []
    partial_set = set()
    successes = 0
    total_partials = 0

    while successes < num_successful_sequences:
        full = None
        for _ in range(max_partial_tries_per_seq):
            p0 = random_partial_sequence(r, partial_flat_len, rng=rng)
            full = extend_to_full_turyn_sat(p0, r, varmap, cnf, rng=rng)
            if full is not None:
                break
        if full is None:
            print(f"Stopped: no completion after {max_partial_tries_per_seq} tries (successes={successes})")
            break

        canon = [list(row) for row in full]
        canonicalize_seq(canon)
        canon_tuples = seq_to_tuples(canon)
        new_pairs = create_partials(canon_tuples, r)
        added = 0
        for partial in new_pairs:
            partial_seq, next_bit = partial
            key = create_partial_str_next_bit(partial_seq, next_bit)
            if key not in partial_set:
                partial_set.add(key)
                dataset.append(partial)
                added += 1
        total_partials += len(new_pairs)
        successes += 1
        print(
            f"sequence {successes}: new_unique_pairs={added}, "
            f"dataset_size={len(dataset)}"
        )

    return dataset


dataset_sat = generate_dataset_sat(
    R,
    PARTIAL_FLAT_LEN,
    num_successful_sequences=4,
    max_partial_tries_per_seq=100,
    rng=random,
)
print(f"\nFinal dataset size (unique partial+next): {len(dataset_sat)}")
if dataset_sat:
    p, nb = dataset_sat[-1]
    print("Last sample next_bit:", nb)
    print("Last partial row lengths:", [len(x) for x in p])

sequence 1: new_unique_pairs=28, dataset_size=28
sequence 2: new_unique_pairs=26, dataset_size=54
sequence 3: new_unique_pairs=24, dataset_size=78
sequence 4: new_unique_pairs=25, dataset_size=103

Final dataset size (unique partial+next): 103
Last sample next_bit: 1
Last partial row lengths: [7, 7, 7, 6]


In [4]:
from sequence_generator_sat import generate_dataset_sat

dataset = generate_dataset_sat(r=7, partial_flat_len=4, limit=2000, max_partial_tries_per_seq=10, verbose=True)
print(len(dataset))

collected 1: canonical rowsums=(-1, 3, 3, 3)
collected 2: canonical rowsums=(-1, 3, 3, 3)
collected 3: canonical rowsums=(1, 1, -1, 5)
collected 4: canonical rowsums=(1, 3, 3, 3)
collected 5: canonical rowsums=(1, 3, 3, -3)
collected 6: canonical rowsums=(-1, 1, 1, 5)
collected 7: canonical rowsums=(-1, 3, -3, 3)
collected 8: canonical rowsums=(-1, 3, -3, 3)
collected 9: canonical rowsums=(-1, -1, 1, 5)
collected 10: canonical rowsums=(-1, 3, 3, -3)
collected 11: canonical rowsums=(1, 3, 3, 3)
collected 12: canonical rowsums=(-1, -3, 3, 3)
collected 13: canonical rowsums=(-1, 1, 1, 5)
collected 14: canonical rowsums=(1, 3, 3, 3)
collected 15: canonical rowsums=(1, 1, -1, 5)
collected 16: canonical rowsums=(1, 3, 3, -3)
collected 17: canonical rowsums=(1, 3, 3, 3)
collected 18: canonical rowsums=(-1, -1, 1, 5)
collected 19: canonical rowsums=(1, 3, 3, 3)
collected 20: canonical rowsums=(1, 3, 3, -3)
collected 21: canonical rowsums=(1, 3, -3, 3)
collected 22: canonical rowsums=(1, 1, -1,

In [1]:
from sequence_generator import generate_dataset

dataset = generate_dataset(r=7)

48
96
Canonicalizing sequence: [(1, 1, 1, -1, -1, 1, -1), (1, 1, 1, -1, 1, -1, -1), (1, -1, 1, -1, -1, 1, 1), (1, 1, 1, -1, 1, 1, 1)]
Sum of sequence: [1, 1, 1, 5]
Canonicalizing sequence: [(1, 1, 1, -1, -1, 1, -1), (1, 1, 1, -1, 1, -1, -1), (1, 1, -1, -1, 1, -1, 1), (1, 1, 1, -1, 1, 1, 1)]
Sum of sequence: [1, 1, 1, 5]
Canonicalizing sequence: [(1, 1, 1, -1, -1, 1, -1), (1, -1, 1, 1, 1, -1, -1), (1, -1, 1, -1, -1, 1, 1), (1, -1, 1, 1, 1, 1, 1)]
Sum of sequence: [1, 1, 1, 5]
Canonicalizing sequence: [(1, 1, 1, -1, -1, 1, -1), (1, -1, 1, 1, 1, -1, -1), (1, -1, 1, -1, -1, 1, 1), (1, 1, 1, 1, 1, -1, 1)]
Sum of sequence: [1, 1, 1, 5]
Canonicalizing sequence: [(1, 1, 1, -1, -1, 1, -1), (1, -1, 1, 1, 1, -1, -1), (1, 1, -1, -1, 1, -1, 1), (1, -1, 1, 1, 1, 1, 1)]
Sum of sequence: [1, 1, 1, 5]
Canonicalizing sequence: [(1, 1, 1, -1, -1, 1, -1), (1, -1, 1, 1, 1, -1, -1), (1, 1, -1, -1, 1, -1, 1), (1, 1, 1, 1, 1, -1, 1)]
Sum of sequence: [1, 1, 1, 5]
Canonicalizing sequence: [(1, 1, 1, -1, 1, -1,